# 05 - Validaciones Iniciales y Detección de Anomalías (Capa Bronze)

Trabajo Práctico 1 - Sección 4

Este notebook hace **diagnóstico exploratorio de solo lectura** sobre las tablas Bronze creadas en `02_Ingesta_Bronze_RUES.ipynb`, `03_Ingesta_Bronze_TRM.ipynb`, `04_Ingesta_Bronze_CIIU.ipynb` y `06_Ingesta_Bronze_SECOP.ipynb`. No se modifica ni se limpia ningún dato — los hallazgos aquí documentados son insumo para el diseño de la futura **Capa Silver**.

Se valida:

1. Conteo total de registros y volumen de nulos/vacíos por columna.
2. Duplicados exactos.
3. Datos atípicos / outliers específicos del dominio (fechas centinela, fechas futuras, valores fuera de rango).
4. Integridad referencial entre RUES y el catálogo CIIU.
5. Validaciones específicas de SECOP: nulos, duplicados, outliers monetarios, fechas inconsistentes e integridad referencial con RUES.

In [ ]:
%python
from pyspark.sql import functions as F

df_rues = spark.table("Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api")
df_trm = spark.table("Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api")
df_ciiu = spark.table("Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api")
df_secop = spark.table("Datos_Empresas.bronze.DE_Semiestructurado_SECOP_Contratos_Api")

print(f"RUES:  {df_rues.count():,} filas, {len(df_rues.columns)} columnas")
print(f"TRM:   {df_trm.count():,} filas, {len(df_trm.columns)} columnas")
print(f"CIIU:  {df_ciiu.count():,} filas, {len(df_ciiu.columns)} columnas")
print(f"SECOP: {df_secop.count():,} filas, {len(df_secop.columns)} columnas")

---

## 1. Valores nulos / vacíos por columna

En RUES, todas las columnas son texto: se cuenta tanto `NULL` como cadena vacía `""` como "sin dato".

In [ ]:
%python
def contar_nulos_y_vacios(df, columnas_texto=()):
    exprs = []
    for c in df.columns:
        if c in columnas_texto:
            cond = F.col(c).isNull() | (F.trim(F.col(c)) == "")
        else:
            cond = F.col(c).isNull()
        exprs.append(F.count(F.when(cond, c)).alias(c))
    return df.select(exprs)


print("Nulos/vacíos por columna - RUES:")
columnas_texto_rues = [c for c, t in df_rues.dtypes if t == "string"]
display(contar_nulos_y_vacios(df_rues, columnas_texto_rues))

In [ ]:
%python
print("Nulos/vacíos por columna - TRM:")
columnas_texto_trm = [c for c, t in df_trm.dtypes if t == "string"]
display(contar_nulos_y_vacios(df_trm, columnas_texto_trm))

In [ ]:
%python
print("Nulos/vacíos por columna - CIIU:")
columnas_texto_ciiu = [c for c, t in df_ciiu.dtypes if t == "string"]
display(contar_nulos_y_vacios(df_ciiu, columnas_texto_ciiu))

---

## 2. Duplicados exactos

Se comparan las columnas originales de la fuente (sin contar `_ingested_at`, que siempre difiere entre corridas).

In [ ]:
%python
def contar_duplicados_exactos(df, columnas_negocio):
    total = df.count()
    unicos = df.select(columnas_negocio).dropDuplicates().count()
    return total, unicos, total - unicos


columnas_negocio_rues = [c for c in df_rues.columns if c not in ("_ingested_at", "_source")]
total, unicos, duplicados = contar_duplicados_exactos(df_rues, columnas_negocio_rues)
print(f"RUES -> total: {total:,} | únicos: {unicos:,} | duplicados exactos: {duplicados:,}")

columnas_negocio_trm = [c for c in df_trm.columns if c not in ("_ingested_at", "_source")]
total_t, unicos_t, duplicados_t = contar_duplicados_exactos(df_trm, columnas_negocio_trm)
print(f"TRM  -> total: {total_t:,} | únicos: {unicos_t:,} | duplicados exactos: {duplicados_t:,}")

columnas_negocio_ciiu = [c for c in df_ciiu.columns if c not in ("_ingested_at", "_source")]
total_c, unicos_c, duplicados_c = contar_duplicados_exactos(df_ciiu, columnas_negocio_ciiu)
print(f"CIIU -> total: {total_c:,} | únicos: {unicos_c:,} | duplicados exactos: {duplicados_c:,}")

In [ ]:
%python
# Adicional: duplicados por llave de negocio (una empresa no debería repetirse
# con la misma matrícula + cámara de comercio)
duplicados_por_matricula = (
    df_rues.groupBy("codigo_camara", "matricula")
    .count()
    .filter(F.col("count") > 1)
)
print(f"Combinaciones (codigo_camara, matricula) repetidas: {duplicados_por_matricula.count():,}")
display(duplicados_por_matricula.orderBy(F.desc("count")).limit(10))

---

## 3. Datos atípicos / outliers

### 3.1 RUES: fechas centinela, formatos inválidos y fechas futuras

Las fechas de RUES llegan como texto `YYYYMMDD`. Se identifican:
* Valores que no calzan con el patrón de 8 dígitos.
* El valor centinela `99991231` usado por la fuente para "sin fecha de vencimiento".
* Fechas de matrícula posteriores a hoy (inconsistentes con la realidad del negocio).

In [ ]:
%python
hoy = F.date_format(F.current_date(), "yyyyMMdd")

formato_invalido = df_rues.filter(~F.col("fecha_matricula").rlike(r"^\d{8}$"))
print(f"fecha_matricula con formato distinto a 8 dígitos: {formato_invalido.count():,}")

centinela_vigencia = df_rues.filter(F.col("fecha_vigencia") == "99991231")
print(f"fecha_vigencia con valor centinela '99991231' (sin vencimiento): {centinela_vigencia.count():,}")

matricula_futura = df_rues.filter(
    F.col("fecha_matricula").rlike(r"^\d{8}$") & (F.col("fecha_matricula") > hoy)
)
print(f"fecha_matricula en el futuro respecto a hoy: {matricula_futura.count():,}")
display(matricula_futura.select("codigo_camara", "matricula", "fecha_matricula").limit(10))

### 3.2 TRM: valores fuera de rango y rangos de vigencia inválidos

* `valor` (tasa de cambio) no debería ser cero ni negativo.
* `vigenciadesde` no debería ser posterior a `vigenciahasta`.

In [ ]:
%python
valor_invalido = df_trm.filter(F.col("valor").cast("double") <= 0)
print(f"TRM con valor <= 0: {valor_invalido.count():,}")

rango_invalido = df_trm.filter(F.col("vigenciadesde") > F.col("vigenciahasta"))
print(f"TRM con vigenciadesde > vigenciahasta: {rango_invalido.count():,}")

display(df_trm.orderBy(F.desc("vigenciadesde")).limit(5))

---

## 4. Integridad referencial: RUES ↔ Catálogo CIIU

`cod_ciiu_act_econ_pri` en RUES debería existir como `code` en el catálogo CIIU. Se cuentan los códigos de RUES que **no** tienen coincidencia en el catálogo (huérfanos) — esto no bloquea nada en Bronze, pero es clave documentarlo antes de hacer el `JOIN` en Silver.

In [ ]:
%python
codigos_ciiu = df_ciiu.select(F.col("code").alias("code_catalogo")).distinct()

codigos_rues_sin_match = (
    df_rues
    .select("cod_ciiu_act_econ_pri")
    .filter(F.col("cod_ciiu_act_econ_pri").isNotNull() & (F.trim(F.col("cod_ciiu_act_econ_pri")) != ""))
    .distinct()
    .join(codigos_ciiu, F.col("cod_ciiu_act_econ_pri") == F.col("code_catalogo"), "left_anti")
)

total_codigos_distintos = df_rues.select("cod_ciiu_act_econ_pri").distinct().count()
print(f"Códigos CIIU distintos usados en RUES (cod_ciiu_act_econ_pri): {total_codigos_distintos:,}")
print(f"Códigos sin coincidencia en el catálogo CIIU: {codigos_rues_sin_match.count():,}")
display(codigos_rues_sin_match.limit(20))

---

## 5. Validaciones SECOP (Contratos Públicos)

Validaciones de calidad sobre `Datos_Empresas.bronze.DE_Semiestructurado_SECOP_Contratos_Api`. Diagnóstico de solo lectura — no se modifica ningún dato.

### 5.1 Nulos / vacíos por columna

Se cuenta `NULL` y `""` en las columnas de texto. Foco en columnas críticas: `id_contrato`, `documento_proveedor`, `valor_del_contrato`, `fecha_de_firma`, `estado_contrato`.

In [ ]:
%python
print("Nulos/vacíos por columna - SECOP:")
cols_texto_secop = [c for c, t in df_secop.dtypes if t == "string"]
display(contar_nulos_y_vacios(df_secop, cols_texto_secop))

### 5.2 Duplicados exactos

Compara todas las columnas de negocio (excluyendo `_ingested_at` y `_source`). Si el total coincide con los únicos, no hay filas repetidas.

In [ ]:
%python
cols_neg_secop = [c for c in df_secop.columns if c not in ("_ingested_at", "_source")]
t, u, d = contar_duplicados_exactos(df_secop, cols_neg_secop)
print(f"SECOP -> total: {t:,} | únicos: {u:,} | duplicados exactos: {d:,}")

### 5.3 Outliers en valores monetarios

`valor_del_contrato` no debería ser negativo ni cero. Se revisan los montos extremos (top 10) para detectar valores atípicos.

In [ ]:
%python
df_secop_v = df_secop.withColumn("valor_num", F.col("valor_del_contrato").cast("double"))

neg = df_secop_v.filter(F.col("valor_num") < 0)
cero = df_secop_v.filter(F.col("valor_num") == 0)
print(f"valor_del_contrato < 0: {neg.count():,}")
print(f"valor_del_contrato = 0: {cero.count():,}")
display(df_secop_v.select("id_contrato", "valor_del_contrato", "estado_contrato").orderBy(F.desc("valor_num")).limit(10))

### 5.4 Fechas inconsistentes

* `fecha_de_inicio_del_contrato` no debería ser posterior a `fecha_de_fin_del_contrato`.
* `fecha_de_firma` no debería estar en el futuro.

In [ ]:
%python
inicio_mayor_fin = df_secop.filter(F.col("fecha_de_inicio_del_contrato") > F.col("fecha_de_fin_del_contrato"))
print(f"fecha_inicio > fecha_fin: {inicio_mayor_fin.count():,}")

firma_futura = df_secop.filter(F.to_date(F.col("fecha_de_firma")) > F.current_date())
print(f"fecha_de_firma en el futuro: {firma_futura.count():,}")
display(firma_futura.select("id_contrato", "fecha_de_firma", "estado_contrato").limit(10))

### 5.5 Integridad referencial SECOP ↔ RUES

Verifica cuántos `documento_proveedor` de SECOP tienen coincidencia con `numero_identificacion` en RUES. Esta es la llave de unión futura en la Capa Silver.

In [ ]:
%python
ids_rues = df_rues.select(F.col("numero_identificacion").alias("id_rues")).distinct()
prov_secop = df_secop.select(F.col("documento_proveedor").alias("id_secop")).distinct().filter(F.col("id_secop").isNotNull())

total_prov = prov_secop.count()
con_match = prov_secop.join(ids_rues, F.col("id_secop") == F.col("id_rues"), "inner").count()
sin_match = total_prov - con_match

print(f"Proveedores únicos en SECOP: {total_prov:,}")
print(f"Con match en RUES: {con_match:,} ({con_match/total_prov*100:.1f}%)")
print(f"Sin match en RUES: {sin_match:,} ({sin_match/total_prov*100:.1f}%)")

---

## Resumen de hallazgos (para la Capa Silver)

_Completar tras ejecutar las celdas anteriores con los números reales obtenidos:_

| Hallazgo | Fuente | Cantidad | Acción sugerida en Silver |
|---|---|---|---|
| Nulos/vacíos por columna | RUES / TRM / CIIU / SECOP | _ver salida_ | Definir reglas de completitud mínima por columna crítica |
| Duplicados exactos | RUES / TRM / CIIU / SECOP | _ver salida_ | `dropDuplicates()` en Silver, nunca en Bronze |
| `fecha_vigencia = 99991231` | RUES | _ver salida_ | Convertir a `NULL` explícito ("sin vencimiento") al tipar como `DATE` |
| Fechas con formato inválido / futuras | RUES | _ver salida_ | Cuarentena o corrección en Silver, no en Bronze |
| `valor <= 0` en TRM | TRM | _ver salida_ | Investigar y excluir en Silver si son errores de origen |
| Códigos CIIU de RUES sin match en el catálogo | RUES ↔ CIIU | _ver salida_ | Decidir si se mapean manualmente, se dejan como "no clasificado" o se excluyen del análisis por sector en Silver |
| `valor_del_contrato <= 0` en SECOP | SECOP | _ver salida_ | Investigar y excluir en Silver si son errores de origen |
| Fechas inconsistentes (`fecha_inicio > fecha_fin`, firma futura) | SECOP | _ver salida_ | Cuarentena o corrección en Silver, no en Bronze |
| Proveedores SECOP sin match en RUES | SECOP ↔ RUES | _ver salida_ | Decidir si son proveedores extranjeros/personas naturales sin RUES, o errores de digitación a corregir en Silver |

> Importante: ninguna de estas anomalías se corrige en este notebook. La Capa Bronze conserva el dato **tal cual llegó de la fuente**, cumpliendo el principio de inmutabilidad.